# Lezione 9: Metriche di Valutazione per Singola Cellula
**File**: [`evaluate.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/evaluate.py)  
**Ruolo nel progetto**: Fornisce le metriche statistiche e distribuzionali per quantificare la fedeltà delle cellule generate rispetto alle cellule reali osservate.


---
## 1. La Sfida: Valutare Distribuzioni Cellulari ad Alta Dimensione

Nel sequenziamento scRNA-seq, non possiamo valutare il modello confrontando singole cellule una a una (perché la misurazione distrugge la cellula). Dobbiamo invece confrontare l'intera **popolazione predetta** $P$ con l'intera **popolazione reale** $Q$!

Le metriche canoniche nella letteratura bioinformatica moderna sono:

### 1.1 Energy Distance ($D_E$)
È una distanza tra distribuzioni di probabilità multivariate definita come:
$$D_E(P, Q) = 2 \, \mathbb{E}_{X, Y}[\|X - Y\|] - \mathbb{E}_{X, X'}[\|X - X'\|] - \mathbb{E}_{Y, Y'}[\|Y - Y'\|]$$
- $D_E(P, Q) \ge 0$, e vale $0$ **se e solo se** le distribuzioni $P$ e $Q$ sono identiche!
- È molto più robusta dell'MSE perché valuta non solo la media, ma anche la dispersione, la covarianza e le forme multimodali.

### 1.2 Correlazione di Pearson sui Profili Medi
Valuta se la gerarchia di espressione dei geni (quali geni sono alti e quali bassi) è preservata fedelmente:
$$r = \frac{\sum_g (\bar{x}_{\text{pred}, g} - \mu_{\text{pred}})(\bar{x}_{\text{true}, g} - \mu_{\text{true}})}{\sigma_{\text{pred}} \sigma_{\text{true}}}$$

### 1.3 Distanza di Wasserstein-1 Marginale
Media della distanza Earth Mover's sui singoli istogrammi dei geni:


---
## 2. Setup e Import


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from evaluate import compute_energy_distance, compute_evaluation_metrics
print('Modulo evaluate caricato!')


---
## 3. Comportamento dell'Energy Distance su Casi di Test

Verifichiamo le proprietà matematiche dell'Energy Distance:
1. Due campioni dalla stessa distribuzione $\implies D_E \approx 0$.
2. Due distribuzioni separate da un offset $\implies D_E > 0$ cresce con la distanza.


In [ ]:
torch.manual_seed(42)
N, G = 200, 10

# Caso 1: Stessa distribuzione gaussiana
p1 = torch.randn(N, G)
p2 = torch.randn(N, G)
e_dist_same = compute_energy_distance(p1, p2)
print(f'Energy Distance (stessa distribuzione): {e_dist_same:.5f} (molto vicina a 0!)')

# Caso 2: Distribuzione traslata di +2.0
p_shifted = torch.randn(N, G) + 2.0
e_dist_shifted = compute_energy_distance(p1, p_shifted)
print(f'Energy Distance (distribuzione traslata): {e_dist_shifted:.5f}')

assert e_dist_shifted > e_dist_same, 'La distanza deve aumentare con la separazione!'


---
## 4. Calcolo del Set Completo di Metriche

Proviamo `compute_evaluation_metrics` simulando una previsione generata e la ground truth reale.


In [ ]:
target_gene = 3
x_true = torch.rand(150, 15) * 2.0
# Simuliamo una predizione quasi corretta con leggero rumore
x_pred = x_true + 0.1 * torch.randn_like(x_true)

metrics = compute_evaluation_metrics(x_pred, x_true, target_gene_idx=target_gene)
for k, v in metrics.items():
    print(f'{k:20s}: {v:.4f}')
